# Key Encapsulation Mechanism (KEM)

References: 

- [wikipedia](https://en.wikipedia.org/wiki/Key_encapsulation_mechanism)

A KEM consists of three algorithms:

- **Key Generation**:  ${\displaystyle ({\mathit {pk}},{\mathit {sk}}):=\operatorname {Keygen} ()}$, takes no inputs and returns a pair of a public key ${\displaystyle {\mathit {pk}}}$ and a private key ${\displaystyle {\mathit {sk}}}$.

- **Encapsulation**: ${\displaystyle (k,c):=\operatorname {Encap} ({\mathit {pk}})}$, takes a public key ${\displaystyle {\mathit {pk}}}$, randomly chooses a secret key ${\displaystyle k}$, and returns ${\displaystyle k}$ along with its encapsulation ${\displaystyle c}$.

- **Decapsulation**: ${\displaystyle (k):=\operatorname {Decap} ({\mathit {sk}, \mathit{c}})}$, takes a private key ${\displaystyle {\mathit {sk}}}$ and an encapsulation ${\displaystyle c'}$, and either returns an encapsulated secret key ${\displaystyle k'}$ or fails, sometimes denoted by returning ${\displaystyle \bot }$.


The figure below show the difference between Public-Key Encryption (A) and Key Encapsulation Mechanism (B).

<div style="display:flex; text-align:center; justify-content: center; gap: 30px;">

  <div style="width: 250px; text-align: center;">
    <p><b>(A) PKE </b></p>
    <img src="imgs/PKE.png" style="width: 200; height: auto;">
  </div>

  <div style="width: 225px; text-align: center;">
    <p><b>(B) KEM </b></p>
    <img src="imgs/KEM.png" style="width: 200; height: auto;">
  </div>

</div>



In [13]:
from IPython.display import IFrame
from kem import print_kem_table

print_kem_table()
IFrame(src="https://pqshield.github.io/nist-sigs-zoo/kems", width=800, height=600)

 AVAILABLE KEMS


Algorithm,NIST Level,Public key (bytes),Secret key (bytes),Ciphertext (bytes),Shared secret (bytes)
ML-KEM-512,1,800,"1,632",768,32
ML-KEM-768,3,"1,184","2,400","1,088",32
ML-KEM-1024,5,"1,568","3,168","1,568",32
BIKE-L1,1,"1,541","5,223","1,573",32
BIKE-L3,3,"3,083","10,105","3,115",32
BIKE-L5,5,"5,122","16,494","5,154",32
HQC-1,1,"2,241","2,321","4,433",32
HQC-3,3,"4,514","4,602","8,978",32
HQC-5,5,"7,237","7,333","14,421",32
FrodoKEM-640-AES,1,"9,616","19,888","9,752",16


In [17]:
import oqs

def short_hex(data, n=40):
    hex_data = data.hex()
    return hex_data[:n * 2] + "..."


BLUE = "\033[94m"
RED = "\033[91m"
GREEN = "\033[92m"
RESET = "\033[0m"

KEM = "ML-KEM-512"
kem = oqs.KeyEncapsulation(KEM)
bob_public_key = kem.generate_keypair()


print("[BOB] Generata la coppia di chiavi")
print(f"[BOB] Public key: {BLUE}0x{short_hex(bob_public_key)}{RESET}")

print()
print("[BOB]   ------ pk ----->  [ALICE]")
print()

print("[ALICE] Ricevuta la public key di Bob")
print("[ALICE] Eseguo encapsulation...")

ciphertext, alice_shared_secret = kem.encap_secret(bob_public_key)

print(f"[ALICE] Ciphertext: {RED}0x{short_hex(ciphertext)}{RESET}")
print(f"[ALICE] Shared secret: {GREEN}0x{alice_shared_secret.hex()}{RESET}")


print()
print("[BOB]   <------ ct -----  [ALICE]")
print()

print("[BOB] Ricevuto il ciphertext")
print("[BOB] Eseguo decapsulation...")

bob_shared_secret = kem.decap_secret(ciphertext)

print(f"[BOB] Shared secret: {GREEN}0x{bob_shared_secret.hex()}{RESET}")
print()


# ============================================================
# VERIFICA
# ============================================================

if alice_shared_secret == bob_shared_secret:
    print("✓ Alice e Bob possiedono la stessa shared secret")
else:
    print("✗ Errore: le shared secret sono diverse")


ciphertext_modified = bytearray(ciphertext)

# Modifichiamo il primo byte
ciphertext_modified[0] ^= 1

ciphertext_modified = bytes(ciphertext_modified)

print("[ATTACKER] ⚠ Ciphertext modificato!")
print(f"[ATTACKER] Originale: {ciphertext.hex()[:32]}...")
print(f"[ATTACKER] Modificato: {ciphertext_modified.hex()[:32]}...")

secret_modified = kem.decap_secret(ciphertext_modified)

print("[BOB] Decapsulation del ciphertext modificato")
print(f"[BOB] Shared secret: {secret_modified.hex()}")

print("\nShared secret originale == modificata:",
      alice_shared_secret == secret_modified)
    

[BOB] Generata la coppia di chiavi
[BOB] Public key: 0x3bb7bca148a9659069d540391d21813d602a3a192e8b086f7fa77c32181097bb4a60823a1f720358...

[BOB]   ------ pk ----->  [ALICE]

[ALICE] Ricevuta la public key di Bob
[ALICE] Eseguo encapsulation...
[ALICE] Ciphertext: 0x781d8bf1880cb388896c24a04b5d2f27fc327e889f9e09368ba25739bef8900ceab8bdf0491d7f31...
[ALICE] Shared secret: 0xedee6971ee85a3a4accd813955024d371a23d346984275684eb03cb770ec03f0

[BOB]   <------ ct -----  [ALICE]

[BOB] Ricevuto il ciphertext
[BOB] Eseguo decapsulation...
[BOB] Shared secret: 0xedee6971ee85a3a4accd813955024d371a23d346984275684eb03cb770ec03f0

✓ Alice e Bob possiedono la stessa shared secret
[ATTACKER] ⚠ Ciphertext modificato!
[ATTACKER] Originale: 781d8bf1880cb388896c24a04b5d2f27...
[ATTACKER] Modificato: 791d8bf1880cb388896c24a04b5d2f27...
[BOB] Decapsulation del ciphertext modificato
[BOB] Shared secret: 8d1f260f6af27ffad88703cc192ae694e99809183761daadb1c83289a07f5746

Shared secret originale == modificata: F

Determinismo della decapsulation — invia lo stesso ciphertext due volte a decap_secret: il segreto restituito è identico (la decap è deterministica, a differenza dell'encap che è randomizzata). Buono spunto di discussione: un ciphertext è "replayable" a livello di primitiva — è il protocollo sopra (TLS, ecc.) che deve garantire freschezza con nonce/timestamp, non il KEM da solo.

# Digital Signature Scheme (DSS)

- Keygen
- Sign
- Verify  

In [12]:
from dss import print_dss_table
from IPython.display import IFrame

print_dss_table()
IFrame(src="https://pqshield.github.io/nist-sigs-zoo", width=800, height=600)

 AVAILABLE DIGITAL SIGNATURE SCHEMES


Algorithm,NIST Level,Public key (bytes),Secret key (bytes),Signature (bytes)
ML-DSA-44,2,"1,312","2,560","2,420"
ML-DSA-65,3,"1,952","4,032","3,309"
ML-DSA-87,5,"2,592","4,896","4,627"
MAYO-1,1,"1,420",24,454
MAYO-2,1,"4,912",24,186
MAYO-3,3,"2,986",32,681
MAYO-5,5,"5,554",40,964
SLH_DSA_PURE_SHA2_128S,1,32,64,"7,856"
SLH_DSA_PURE_SHA2_192S,3,48,96,"16,224"
SLH_DSA_PURE_SHA2_256S,5,64,128,"29,792"
